<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/homework_solutions/04a_control_barrier_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Barrier Function Filter

In this problem, you will implement a CBF-QP safety filter.

To start of, we will restrict ourselves to a simple 1D system so that you can verify your results analytically if needed.


 but note that the theory extends to higher dimensional problems, and try to keep your code general so that it could still work on a different system and choice of CBF.

Consider the following 1D single integator dynamics $\dot{x} = u$ and let $b(x) = x^2 - 1$.
In this problem, $b(x)=x^2-1$ defines a safety condition (e.g., a set where $b(x)\leq 0$ is safe), and $f(x, u) = u$ describes the system's dynamics.
The Lie derivative $\nabla b(x)^T f(x, u)$ therefore describes how $b(x)$ changes due to the control input $u$, and is central to Control Barrier Function (CBF) approaches for enforcing safety constraints through feedback or filtering.

The Lie derivative of a function $b(x)$ with respect to the dynamics $f(x, u)$ measures how $b(x)$ changes as the system evolves according to those dynamics.

Mathematically, the Lie derivative is given by $\nabla b(x)^T f(x, u)$, where $\nabla b(x)$ is the gradient of $b$ with respect to $x$.
This expression tells us the instantaneous rate of change of $b(x)$ when the state $x$ flows along the vector field defined by $f(x, u)$.



In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import functools
import cvxpy as cp
import numpy as np



In [ ]:
def f(x, u):
    return u

def b(x):
    return x**2 - 1


Plot the control barrier function

In [ ]:

x_vals = jnp.linspace(-2.5, 2.5, 200)
b_vals = b(x_vals)

plt.figure(figsize=(6,4))
plt.plot(x_vals, b_vals, label="$b(x) = x^2 - 1$")
plt.axhline(0, color='k', linestyle='--', linewidth=1)
plt.xlabel("$x$")
plt.ylabel("$b(x)$")
plt.title("CBF: $b(x) = x^2 - 1$")
plt.grid(True)
plt.legend()
plt.show()

### (a) Compute the Lie derivative $\nabla b(x)^Tf(x,u)$

Verify the correctness of your implementation by deriving the analytic expression and comparing some values.



In [ ]:
def _lie_derivative(dynamics_func, cbf_func, state, control):

    # TODO: compute the Lie derivative using JAX
    ###### add your code here
    # HINT: take a look at jax.jvp
    # tangent = ...
    # return ...
    ###### end of add your code here

    ###### start of solutions
    tangent = dynamics_func(state, control)
    return jax.jvp(cbf_func, (state,), (tangent,))[1]
    ###### end of solutions

def analytic_lie_derivative(state, control):

    # TODO: compute the Lie derivative analytically
    ###### add your code here
    # tangent = ...
    # return ...
    ###### end of add your code here

    ##### start of solutions
    return 2 * state * control
    ##### end of solutions
lie_derivative = functools.partial(_lie_derivative, f, b)



In [ ]:
# check if the implementation is correct
key = jax.random.PRNGKey(0)
bs = 128
state = jax.random.uniform(key, (bs, 1))
control = jax.random.uniform(key, (bs, 1))
jnp.allclose(jax.vmap(lie_derivative, in_axes=(0, 0))(state, control), analytic_lie_derivative(state, control))


### (b) Solving the CBF-QP

Consider a scenario where your desired control input is $u_\mathrm{des} = 0.5$, aiming to move in the positive $x$-direction at constant velocity.
However, a safety constraint requires that $x^2 \geq 1$ at all times; this means the system must stay at least 1 unit away from the origin.

The Control Barrier Function (CBF) safety filter will adjust the control input to be as close as possible to $u_\mathrm{des}$, but only if it can do so while satisfying the CBF constraint.

The CBF-QP (Quadratic Program) can be formulated as follows:
$$
u_\mathrm{safe}(x) = \underset{u}{\text{argmin}} \| u - u_\mathrm{des}\|_2^2 \quad \text{subject to} \quad \nabla b(x)^\top f(x,u) \geq -\alpha\big(b(x)\big)
$$
In this optimization problem, the objective is quadratic and the constraint is linear in $u$.
For simplicity, assume there are no other constraints on the control input.

Let the class-$\mathcal{K}$ function $\alpha(z) = az$ with $a = 0.5$.

**Task:**  
Use `cvxpy` to solve the CBF-QP for the following values of $x$: $x = -3$, $x = -2$, and $x = -1.1$.

Report the corresponding safe control values $u_\mathrm{safe}(x)$ for each state.

**Hint:**  
The `Parameter` class in `cvxpy` is convenient for updating problem parameters, such as the state $x$, without reconstructing the entire optimization problem.


In [ ]:
# TODO: implement the CBF-QP safety filter
##### add your code here
# NOTE: cvxpy does not support jax.numpy arrays, so we need to convert to numpy arrays
# set up CVXPY variables and parameters
# solve the CBF-QP for each initial state
##### end of add your code here


##### start of solutions
u = cp.Variable(1)
x = cp.Parameter(1)
linear_term = cp.Parameter(1)
cbf_value = cp.Parameter(1)
a = cp.Parameter(1)
u_des = cp.Parameter(1)
objective = cp.Minimize((u - u_des)**2)
constraints = [linear_term @ u + a @ cbf_value >= 0]
problem = cp.Problem(objective, constraints)

a.value = np.array([0.5])
u_des.value = np.array([0.5])
xs = jnp.array([-3., -2., -1.1])
for x_val in xs:
    linear_term.value = np.array(jax.grad(b)(x_val))[None]
    cbf_value.value = np.array(b(x_val))[None]
    problem.solve()
    print("x = {:.2f}, u_safe = {:.2f}".format(x_val, u.value[0]))
##### end of solutions

### (c) Applying the CBF Safety Filter
In this task, you will investigate the effect of the control barrier function (CBF) safety filter on the system dynamics.

**Instructions:**
- Simulate the system starting from the initial state $x = -5$, using a desired control input $u_\mathrm{des} = 0.5$.
- At each step, apply the CBF safety filter to compute the safe control $u_\mathrm{safe}$ before updating the state.
- Consider four values of the parameter $a$ in the class-$\mathcal{K}$ function $\alpha(z) = az$, specifically: $a = 2$, $a = 1$, $a = 0.5$, and $a = 0.1$.
- For each value of $a$, simulate and plot the resulting state and control input trajectories over time.

**Simulation details:**
- Use a time step of $\Delta t = 0.05$.
- Run the simulation for 500 time steps.

**Discussion:**
- Compare the trajectories for the different values of $a$.
- Comment on how the choice of $a$ affects the system behavior and the conservatism of the safety filter.
- What practical role does the parameter $a$ play in shaping the system's response?

*Note:* While CBF theory is designed for continuous-time systems, we are applying it here in discrete time for simulation purposes. This can lead to certain practical challenges (see [this paper](https://arxiv.org/abs/2404.12329) for more details), but using a sufficiently small time step is generally effective for our purposes.


In [ ]:
# TODO: implement the CBF-QP safety filter
##### add your code here
# you may need come helper functions
# you may need to loop through the different values of a
# you will need to write your own plotting code
##### end of add your code here

##### start of solutions
def step(state, control, dt):
    return state + control * dt

def simulate(initial_state, desired_control, a_parameter, dt, num_steps):
    xs = [initial_state]
    us = []
    a.value = np.array([a_parameter])
    u_des.value = np.array([desired_control])
    for _ in range(num_steps):
        x_val = xs[-1]
        linear_term.value = np.array(jax.grad(b)(x_val))[None]
        cbf_value.value = np.array(b(x_val))[None]
        problem.solve()
        u_safe = jnp.array(u.value[0])
        x_next = step(x_val, u_safe, dt)
        xs.append(x_next)
        us.append(u_safe)
    return jnp.stack(xs), jnp.stack(us)

as_val = [2, 1, 0.5, 0.1]
xs = []
us = []
for a_val in as_val:
    xs_, us_     = simulate(jnp.array(-5.), jnp.array(0.5), a_val, 0.05, 500)
    xs.append(xs_)
    us.append(us_)
xs = jnp.stack(xs, axis=1).T
us = jnp.stack(us, axis=1).T

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.title('Trajectories')
plt.xlabel('Time step')
plt.ylabel('x')
plt.plot(xs.T)
plt.grid(alpha=0.3)
plt.legend(['a = {}'.format(a_val) for a_val in as_val])

plt.subplot(1,2,2)
plt.title('Control inputs')
plt.xlabel('Time step')
plt.ylabel('u')
plt.plot(us.T)
plt.grid(alpha=0.3)
plt.legend(['a = {}'.format(a_val) for a_val in as_val])

plt.tight_layout()
##### end of solutions

